<a href="https://colab.research.google.com/github/shokuhatsugi/computer-vision-etl-pipeline/blob/main/cv_etl_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. Install required packages in Google Colab
!pip install -q prefect opencv-python tensorflow numpy pillow

# 2. Imports
import os
import glob
import numpy as np
import cv2
import tensorflow as tf
from prefect import task, flow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.4/63.4 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 60.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 71.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 279.5/279.5 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.4/322.4 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.4/169.4 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/6

In [2]:
RAW_DATA_DIR = "./raw_camera_feed"
os.makedirs(RAW_DATA_DIR, exist_ok=True)

def generate_sample_images(num_images=8):
    """Generates mock camera images with random dimensions to test standardization."""
    for i in range(num_images):
        h, w = np.random.randint(300, 900), np.random.randint(300, 900)
        img = np.random.randint(0, 256, (h, w, 3), dtype=np.uint8)
        cv2.imwrite(os.path.join(RAW_DATA_DIR, f"cam_feed_{i:03d}.jpg"), img)

generate_sample_images(8)
print(f"Generated sample raw camera images in: '{RAW_DATA_DIR}'")

Generated sample raw camera images in: './raw_camera_feed'


In [3]:
@task(name="Extract Raw Paths", retries=2)
def extract_image_paths(directory: str) -> list:
    """Extract stage: Discovers raw camera image file paths."""
    paths = glob.glob(os.path.join(directory, "*.jpg")) + glob.glob(os.path.join(directory, "*.png"))
    print(f"[EXTRACT] Discovered {len(paths)} raw image files.")
    return paths

In [4]:
@task(name="Transform Image (OpenCV Preprocessing)", retries=2)
def preprocess_image(image_path: str, target_dim=(224, 224)) -> tuple:
    """
    Transform stage:
    1. BGR to RGB conversion
    2. Resizing to target dimensions (224x224)
    3. Pixel normalization (0.0 to 1.0)
    """
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f"Unable to read image file: {image_path}")

    # 1. OpenCV BGR to standard RGB conversion
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # 2. Resizing to uniform 224x224 dimension
    img_resized = cv2.resize(img_rgb, target_dim, interpolation=cv2.INTER_AREA)

    # 3. Min-Max Normalization (0 to 255 -> 0.0 to 1.0)
    img_normalized = img_resized.astype(np.float32) / 255.0

    filename = os.path.basename(image_path)
    return img_normalized, filename

In [5]:
@task(name="Load to TensorFlow Dataset")
def load_to_tf_dataset(preprocessed_data: list, batch_size: int = 4) -> tf.data.Dataset:
    """
    Load stage: Converts preprocessed numpy arrays into an optimized tf.data.Dataset pipeline.
    """
    images = np.array([item[0] for item in preprocessed_data], dtype=np.float32)
    filenames = [item[1] for item in preprocessed_data]

    # TensorFlow tensor dataset creation
    dataset = tf.data.Dataset.from_tensor_slices((images, filenames))

    # Batching and GPU prefetching optimization
    dataset = dataset.batch(batch_size).prefetch(buffer_size=tf.data.AUTOTUNE)
    print(f"[LOAD] Successfully loaded dataset into tf.data pipeline (Batch Size: {batch_size}).")
    return dataset

In [6]:
@flow(name="Automated Vision Data Pipeline")
def computer_vision_etl_flow(raw_dir: str, target_dim=(224, 224), batch_size=4):
    """Orchestrates the Extraction, Transformation, and TensorFlow Loading tasks."""
    # 1. Extract
    raw_paths = extract_image_paths(raw_dir)

    # 2. Transform (Looping over images)
    processed_records = []
    for path in raw_paths:
        processed_img, fname = preprocess_image(path, target_dim=target_dim)
        processed_records.append((processed_img, fname))

    # 3. Load
    tf_dataset = load_to_tf_dataset(processed_records, batch_size=batch_size)
    return tf_dataset

In [7]:
# Execute the complete flow
tf_dataset = computer_vision_etl_flow(RAW_DATA_DIR, target_dim=(224, 224), batch_size=4)

# Verify transformed tensors
for batch_images, batch_filenames in tf_dataset.take(1):
    print("\n================ Verification Summary ================")
    print(f"Batch Tensor Shape : {batch_images.shape}")                  # Output: (4, 224, 224, 3)
    print(f"Pixel Min Value    : {tf.reduce_min(batch_images).numpy():.2f}") # Output: 0.00
    print(f"Pixel Max Value    : {tf.reduce_max(batch_images).numpy():.2f}") # Output: 1.00
    print(f"Data Type          : {batch_images.dtype}")                  # Output: <dtype: 'float32'>
    print("======================================================")

INFO:prefect:Starting temporary server on http://127.0.0.1:8363
See https://docs.prefect.io/v3/concepts/server#how-to-guides for more information on running a dedicated Prefect server.
INFO:prefect.flow_runs:Beginning flow run 'dandelion-mosquito' for flow 'Automated Vision Data Pipeline'
INFO:prefect.task_runs:Finished in state Completed()
INFO:prefect.task_runs:Finished in state Completed()
INFO:prefect.task_runs:Finished in state Completed()
INFO:prefect.task_runs:Finished in state Completed()
INFO:prefect.task_runs:Finished in state Completed()
INFO:prefect.task_runs:Finished in state Completed()


[EXTRACT] Discovered 8 raw image files.


INFO:prefect.task_runs:Finished in state Completed()
INFO:prefect.task_runs:Finished in state Completed()
INFO:prefect.task_runs:Finished in state Completed()
INFO:prefect.task_runs:Finished in state Completed()


[LOAD] Successfully loaded dataset into tf.data pipeline (Batch Size: 4).


INFO:prefect.flow_runs:Finished in state Completed()



================ Verification Summary ================
Batch Tensor Shape : (4, 224, 224, 3)
Pixel Min Value    : 0.13
Pixel Max Value    : 0.85
Data Type          : <dtype: 'float32'>
